# _Second step_

It is essential to verify that the predictors selected from the literature exhibit significant predictive power and are sufficiently independent to prevent multicollinearity, thereby ensuring the robustness of the model.

### Objective
Before training the Deep Convolutional Neural Network (CNN), it is critical to understand the statistical relationships between our environmental drivers (features) and actual fire ignitions (target). 

This notebook calculates the **Spearman Rank Correlation** across millions of spatial pixels. Spearman is chosen over Pearson because environmental and meteorological relationships (like vapor pressure deficit driving fire spread) are often non-linear and monotonic rather than strictly linear.

### Scientific Rationale
This analysis serves two mandatory machine learning diagnostics:
1. **Feature Importance:** Does a variable actually correlate with fire events? If a feature shows near-zero correlation across all years, it is adding gradient noise and should be dropped.
2. **Multicollinearity (The Ablation Precursor):** Deep Learning models struggle when fed highly redundant features. If two predictors (e.g., Temperature and Vapor Pressure Deficit) have an extreme positive correlation (e.g., $\rho > 0.85$), they are mathematically fighting for the same gradient updates. This matrix will dictate which variables are "ablated" (removed) in the next phase.

### The Temporal Ablation Windows
Because modern satellite sensors were launched at different times, the dataset is split into three distinct temporal windows. The matrix calculates correlations strictly where valid data overlaps:
* **Window A (2001–2026) | Baseline Core:** Evaluates the fundamental climatology (Temperature, Precipitation, Dewpoint, VPD) alongside static geography (Topography, Infrastructure) and Land Use.
* **Window B (2015–2026) | Soil Moisture Integration:** Introduces SMAP data to evaluate how deep-soil drought memory compares to atmospheric VPD.
* **Window C (2018–2026) | The Emissions Trace:** Introduces Sentinel-5P Carbon Monoxide (CO). CO acts as a chemical tracer for active biomass burning and is expected to have the highest correlation with the MODIS Fire target.

### Output Artifacts
The script flattens the multi-dimensional `.tif` arrays, computes the matrix alongside $p$-values to ensure statistical significance ($p < 0.05$), and exports highly formatted, color-coded **Excel files** for human review and academic publication.

In [1]:
import os
import numpy as np
import rasterio
import pandas as pd
from scipy.stats import spearmanr
from itertools import combinations
from IPython.display import display

# =====================================================================
# SPEARMAN CORRELATION MATRIX (SAVED AS FORMATTED EXCEL)
# =====================================================================

# WSL/Ubuntu Rclone Mount Path
data_dir = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Queimadas_Data"
output_dir = os.path.join(data_dir, "Models and Diagnosis")
os.makedirs(output_dir, exist_ok=True)

N_SAMPLES   = 500_000   # Pooled sample size for computational efficiency
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Variable descriptions for legend
VAR_DESCRIPTIONS = {
    "Fire":     "MODIS Active Fire Ignitions (Target)",
    "Temp":     "ERA5 2m temperature (K)",
    "Precip":   "ERA5 total precipitation (m)",
    "Dewpoint": "ERA5 2m dewpoint temperature (K)",
    "VPD":      "Vapour pressure deficit computed from Temp & Dewpoint (kPa)",
    "EVI":      "MODIS Enhanced Vegetation Index",
    "SMAP":     "SMAP soil moisture (m³/m³)",
    "CO":       "Sentinel‑5P CO column number density (mol/m²)",
    "Roads":    "Euclidean distance to roads (m)",
    "DEM":      "Digital elevation model (m)",
    "Slope":    "Terrain slope (degrees)",
    "Aspect":   "Terrain aspect (degrees clockwise from north)",
}

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def build_common_sample(var_definitions, years, n_samples):
    print("  Pass 1: Verifying coordinate alignment and tracking temporal bounds...")
    raw = {label: [] for label, _ in var_definitions}
    
    for year in years:
        files_exist = True
        for label, pattern in var_definitions:
            fpath = pattern if '{year}' not in pattern else pattern.format(year=year)
            if not os.path.exists(fpath):
                files_exist = False
                break
        
        if not files_exist:
            continue 
            
        year_data = {}
        for label, pattern in var_definitions:
            fpath = pattern if '{year}' not in pattern else pattern.format(year=year)
            with rasterio.open(fpath) as src:
                arr = src.read().astype(np.float32)
                if src.nodata is not None:
                    arr[arr == src.nodata] = np.nan
                year_data[label] = arr
                
        dyn_labels = [l for l, p in var_definitions if '{year}' in p]
        n_days = min(year_data[l].shape[0] for l in dyn_labels) if dyn_labels else 1
            
        for label, pattern in var_definitions:
            is_static = '{year}' not in pattern
            if is_static:
                aligned = np.repeat(year_data[label], n_days, axis=0)
            else:
                aligned = year_data[label][:n_days]
            raw[label].append(aligned.reshape(-1))
            
    if not raw[var_definitions[0][0]]:
        return {}
        
    combined = {label: np.concatenate(raw[label]) for label, _ in var_definitions}
    
    # Enforce intersection logic
    valid = np.ones(combined[var_definitions[0][0]].shape[0], dtype=bool)
    for arr in combined.values():
        valid &= np.isfinite(arr)
        
    valid_idx = np.where(valid)[0]
    print(f"  Intersection absolute coverage: {len(valid_idx):,} valid matrix observations.")
    
    if len(valid_idx) < 100:
        return {}
        
    chosen = rng.choice(valid_idx, size=min(n_samples, len(valid_idx)), replace=False)
    chosen.sort()
    return {label: combined[label][chosen] for label, _ in var_definitions}

def spearman_matrix_with_p(sample_dict):
    labels = list(sample_dict.keys())
    n = len(labels)
    mat = np.zeros((n, n), dtype=np.float64)
    p_mat = np.zeros((n, n), dtype=np.float64)
    
    for i in range(n):
        mat[i, i] = 1.0
        p_mat[i, i] = 0.0
        
    for i, j in combinations(range(n), 2):
        r, p = spearmanr(sample_dict[labels[i]], sample_dict[labels[j]])
        mat[i, j] = mat[j, i] = r
        p_mat[i, j] = p_mat[j, i] = p
        
    return labels, mat, p_mat

def generate_and_save_matrix(labels, mat, p_mat, title, outpath_xlsx, var_descriptions):
    """Create a styled Excel file with the correlation matrix, stars, and legend."""
    n = len(labels)
    # Build the display DataFrame with significance stars
    df_display = pd.DataFrame(index=labels, columns=labels, dtype=str)
    for i in range(n):
        for j in range(n):
            if i == j:
                df_display.iloc[i, j] = "1.000"
            else:
                val = mat[i, j]
                p = p_mat[i, j]
                stars = "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.1 else ""))
                df_display.iloc[i, j] = f"{val:.3f}{stars}"
    
    # Create an Excel writer with xlsxwriter for formatting
    writer = pd.ExcelWriter(outpath_xlsx, engine='xlsxwriter')
    workbook = writer.book
    
    # Write the title and legend separately later, first the matrix
    df_display.to_excel(writer, sheet_name='Correlation Matrix', startrow=2, startcol=0)
    worksheet = writer.sheets['Correlation Matrix']
    
    # --- Format definitions ---
    header_format = workbook.add_format({
        'bold': True,
        'border': 1,
        'bg_color': '#D9E1F2',
        'align': 'center',
        'valign': 'vcenter',
        'text_wrap': True
    })
    diag_format = workbook.add_format({
        'border': 1,
        'bg_color': '#D9D9D9',  # grey diagonal
        'align': 'center',
        'valign': 'vcenter',
        'num_format': '0.000'   # will be overwritten for text cells
    })
    red_bold_format = workbook.add_format({
        'border': 1,
        'bold': True,
        'font_color': 'red',
        'bg_color': '#FFC7CE',
        'align': 'center',
        'valign': 'vcenter'
    })
    blue_italic_format = workbook.add_format({
        'border': 1,
        'italic': True,
        'font_color': 'blue',
        'bg_color': '#BDD7EE',
        'align': 'center',
        'valign': 'vcenter'
    })
    normal_format = workbook.add_format({
        'border': 1,
        'align': 'center',
        'valign': 'vcenter'
    })
    
    # Write column headers (bold)
    for col_idx, label in enumerate(labels):
        worksheet.write(2, col_idx + 1, label, header_format)
    # Write row headers (bold)
    for row_idx, label in enumerate(labels):
        worksheet.write(row_idx + 3, 0, label, header_format)
    
    # Write data with conditional formatting
    for i in range(n):
        for j in range(n):
            cell_value = df_display.iloc[i, j]
            abs_val = abs(mat[i, j])
            if i == j:
                fmt = diag_format
                # Write numeric value but as float to keep format (or write string)
                # The df_display contains "1.000", we'll write it as a number to keep diag_format number style
                worksheet.write_number(i + 3, j + 1, 1.0, diag_format)
            else:
                if abs_val > 0.7:
                    fmt = red_bold_format
                elif abs_val > 0.5:
                    fmt = blue_italic_format
                else:
                    fmt = normal_format
                # Write the string with stars, but using write_string to preserve it
                worksheet.write_string(i + 3, j + 1, cell_value, fmt)
    
    # Set column widths
    worksheet.set_column(0, 0, 12, header_format)
    worksheet.set_column(1, n, 14, normal_format)
    
    # --- Write legend at the bottom ---
    legend_start_row = n + 5
    legend_header_fmt = workbook.add_format({'bold': True, 'font_size': 12})
    legend_text_fmt = workbook.add_format({'font_size': 10})
    
    worksheet.merge_range(legend_start_row, 0, legend_start_row, n, 'LEGEND', legend_header_fmt)
    legend_start_row += 1
    
    legend_items = [
        ('Significance levels:', '*** p<0.01, ** p<0.05, * p<0.1'),
        ('Correlation strength:', 'Bold Red: |ρ| > 0.7  (high redundancy)'),
        ('', 'Italic Blue: 0.5 < |ρ| ≤ 0.7 (moderate)'),
        ('Variable descriptions:', '')
    ]
    for item_title, item_text in legend_items:
        if item_title:
            worksheet.write(legend_start_row, 0, item_title, workbook.add_format({'bold': True}))
        if item_text:
            worksheet.write(legend_start_row, 1, item_text, legend_text_fmt)
        legend_start_row += 1
    
    # Variable descriptions
    for var, desc in var_descriptions.items():
        worksheet.write(legend_start_row, 0, var, workbook.add_format({'bold': True}))
        worksheet.write(legend_start_row, 1, desc, legend_text_fmt)
        legend_start_row += 1
    
    # Freeze panes for headers
    worksheet.freeze_panes(3, 1)
    
    writer.close()
    print(f"  Formatted correlation matrix saved to: {os.path.basename(outpath_xlsx)}")

# ------------------------------------------------------------
# File layer pattern dictionary 
# ------------------------------------------------------------
patterns = {
    "Fire"     : os.path.join(data_dir, 'MODIS_Fire_{year}.tif'),
    "Temp"     : os.path.join(data_dir, 'ERA5_Temp_{year}.tif'),
    "Precip"   : os.path.join(data_dir, 'ERA5_Precip_{year}.tif'),
    "Dewpoint" : os.path.join(data_dir, 'ERA5_Dewpoint_{year}.tif'),
    "VPD"      : os.path.join(data_dir, 'ERA5_VPD_{year}.tif'),
    "EVI"      : os.path.join(data_dir, 'MODIS_EVI_{year}.tif'),
    "SMAP"     : os.path.join(data_dir, 'SMAP_SoilMoisture_{year}.tif'),
    "CO"       : os.path.join(data_dir, 'Sentinel5P_CO_{year}.tif'),
    "Roads"    : os.path.join(data_dir, 'Distance_to_Roads.tif'),
    "DEM"      : os.path.join(data_dir, 'Topography_DEM.tif'),
    "Slope"    : os.path.join(data_dir, 'Topography_Slope.tif'),
    "Aspect"   : os.path.join(data_dir, 'Topography_Aspect.tif'),
}

WINDOWS = [
    {
        "name"   : "Correlation Matrix: Window A (2001–2026 Baseline Core)",
        "years"  : list(range(2001, 2027)),
        "vars"   : ["Fire", "Temp", "Precip", "Dewpoint", "VPD", "EVI", "Roads", "DEM", "Slope", "Aspect"],
        "outfile": os.path.join(output_dir, "Correlation_Window_A.xlsx"),
    },
    {
        "name"   : "Correlation Matrix: Window B (2015–2026 Soil Moisture Added)",
        "years"  : list(range(2015, 2027)),
        "vars"   : ["Fire", "Temp", "Precip", "Dewpoint", "VPD", "EVI", "SMAP", "Roads", "DEM", "Slope", "Aspect"],
        "outfile": os.path.join(output_dir, "Correlation_Window_B.xlsx"),
    },
    {
        "name"   : "Correlation Matrix: Window C (2018–2026 Emissions Trace Optimized)",
        "years"  : list(range(2018, 2027)),
        "vars"   : ["Fire", "Temp", "Precip", "Dewpoint", "VPD", "EVI", "SMAP", "CO", "Roads", "DEM", "Slope", "Aspect"],
        "outfile": os.path.join(output_dir, "Correlation_Window_C.xlsx"),
    },
]

# ------------------------------------------------------------
# Main execution
# ------------------------------------------------------------
for window in WINDOWS:
    print(f"\n{'#'*70}\n Processing Matrix Window: {window['name']}\n{'#'*70}")
    var_defs = [(label, patterns[label]) for label in window['vars']]
    sample_dict = build_common_sample(var_defs, window['years'], N_SAMPLES)

    if len(sample_dict) < 2:
        print(f"  [SKIPPED] Critical files missing for selected timeline window bounds.\n")
        continue

    labels, mat, p_mat = spearman_matrix_with_p(sample_dict)
    generate_and_save_matrix(labels, mat, p_mat, window['name'], window['outfile'], VAR_DESCRIPTIONS)

print("[PROCESS COMPLETE] Correlation matrices saved as formatted Excel files.")


######################################################################
 Processing Matrix Window: Correlation Matrix: Window A (2001–2026 Baseline Core)
######################################################################
  Pass 1: Verifying coordinate alignment and tracking temporal bounds...
  Intersection absolute coverage: 8,368,136 valid matrix observations.
  Formatted correlation matrix saved to: Correlation_Window_A.xlsx

######################################################################
 Processing Matrix Window: Correlation Matrix: Window B (2015–2026 Soil Moisture Added)
######################################################################
  Pass 1: Verifying coordinate alignment and tracking temporal bounds...
  Intersection absolute coverage: 1,551,062 valid matrix observations.
  Formatted correlation matrix saved to: Correlation_Window_B.xlsx

######################################################################
 Processing Matrix Window: Correlation Matrix: Win